In [14]:
import random


class Packet:
    """Simple data container"""
    def __init__(self, source, destination, data):
        self.source = source
        self.destination = destination
        self.data = data
    
    def __str__(self):
        return f"[{self.source} → {self.destination}]: '{self.data}'"


class EndDevice:
    """Computer/Phone - creates and receives data"""
    def __init__(self, name):
        self.name = name
        self.connections = []   # Who am I connected to?
        self.inbox = []        # Messages I received
    
    def connect(self, other):
        """Create physical connection (bidirectional)"""
        if other not in self.connections:
            self.connections.append(other)
            if self not in other.connections:
                other.connections.append(self)
            print(f"[CONNECT] {self.name} ↔ {other.name}")
    
    # ============================================
    # METHOD 1: BROADCAST SEND (for shared media like Hub)
    # ============================================
    def send_broadcast(self, packet):
        """
        Send to ALL connected devices.
        Used when: connected to a Hub (shared medium).
        The receiver must check if packet is for them.
        """
        print(f"\n[BROADCAST SEND] {self.name} sending: {packet}")
        print(f"                 To ALL connections: {[d.name for d in self.connections]}")
        
        for device in self.connections:
            device.receive(packet, self)
    
    # ============================================
    # METHOD 2: DIRECT SEND (for point-to-point links)
    # ============================================
    def send_direct(self, packet, target_device):
        """
        Send to ONE specific device directly.
        Used when: dedicated cable between two devices.
        No need for receiver to check - we know exactly who gets it.
        
        Returns: True if sent successfully, False if target not connected
        """
        print(f"\n[DIRECT SEND] {self.name} sending: {packet}")
        print(f"              Directly to: {target_device.name}")
        
        # Check if we have a direct cable to target
        if target_device not in self.connections:
            print(f"  ✗ ERROR: {target_device.name} is not directly connected to {self.name}!")
            print(f"           Available connections: {[d.name for d in self.connections]}")
            return False
        
        # Send only to that specific device
        target_device.receive_direct(packet, self)
        return True
    
    def receive(self, packet, from_device):
        """
        RECEIVE METHOD 1: For broadcast (shared media).
        Must CHECK if packet is meant for me.
        """
        print(f"  [RECEIVE-BROADCAST] {self.name} got message from {from_device.name}")
        
        if packet.destination == self.name or packet.destination == "ALL":
            print(f"    ✓ ACCEPTED: This message is for me!")
            self.inbox.append(packet)
            return True
        else:
            print(f"    ✗ IGNORED: This is for {packet.destination}, not me")
            return False
    
    def receive_direct(self, packet, from_device):
        """
        RECEIVE METHOD 2: For direct point-to-point.
        No need to check - if we got it, it's ours!
        """
        print(f"  [RECEIVE-DIRECT] {self.name} got message from {from_device.name}")
        print(f"    ✓ ACCEPTED: Direct delivery (no check needed)")
        self.inbox.append(packet)
        return True


class Hub:
    """Hub - broadcasts everything to everyone"""
    def __init__(self, name, ports=4):
        self.name = name
        self.ports = ports
        self.connections = []
    
    def connect(self, device):
        """Plug device into hub"""
        if len(self.connections) >= self.ports:
            print(f"[ERROR] {self.name} full!")
            return
        
        if device not in self.connections:
            self.connections.append(device)
            if self not in device.connections:
                device.connections.append(self)
            port_num = len(self.connections)
            print(f"[CONNECT] {device.name} → {self.name} (Port {port_num})")
    
    def receive(self, packet, from_device):
        """Hub receives and broadcasts to ALL except sender"""
        print(f"\n[HUB] {self.name} received from {from_device.name}")
        print(f"      Broadcasting to {len(self.connections)-1} other ports...")
        
        for device in self.connections:
            if device != from_device:
                print(f"      → To {device.name}")
                device.receive(packet, self)  # Use broadcast receive

In [15]:


def test_direct_connection():
    """
    Test: Two PCs connected directly with a cable.
    Uses send_direct() - no hub, no broadcast, no checking needed!
    
    Topology: PC1 -------- PC2
    """
    print("=" * 60)
    print("TEST 1: DIRECT POINT-TO-POINT CONNECTION")
    print("=" * 60)
    print("Topology: PC1 -------- PC2 (direct cable)")
    print("Method: send_direct() - no hub involved\n")
    
    # Create two PCs
    pc1 = EndDevice("PC1")
    pc2 = EndDevice("PC2")
    
    # Connect them directly (like a crossover cable)
    pc1.connect(pc2)
    
    print(f"\n--- PC1 sends directly to PC2 ---")
    msg = Packet("PC1", "PC2", "Hello PC2!")
    
    # Use DIRECT send (not broadcast!)
    success = pc1.send_direct(msg, pc2)
    
    # Check results
    print(f"\n--- RESULTS ---")
    print(f"PC1 inbox: {len(pc1.inbox)} messages")
    print(f"PC2 inbox: {len(pc2.inbox)} messages")
    
    if success and len(pc2.inbox) == 1 and len(pc1.inbox) == 0:
        print("\n✓ TEST PASSED: Direct send worked!")
        print("  - PC2 received the message")
        print("  - PC1 did not receive it (no loopback)")
        print("  - No unnecessary broadcast")
    else:
        print("\n✗ TEST FAILED")
    
    # Try sending to someone not connected
    print(f"\n--- Testing error case: send to unconnected device ---")
    pc3 = EndDevice("PC3")
    msg2 = Packet("PC1", "PC3", "Hello?")
    pc1.send_direct(msg2, pc3)  # Should fail - not connected!


In [16]:
# ============================================================
# TEST CASE 2: Star with Hub (Broadcast method)
# ============================================================

def test_star_with_hub():
    """
    Test: 5 PCs connected to a Hub.
    Uses send_broadcast() - hub repeats to everyone!
    
    Topology:
         PC1
          |
         PC2
          |
    PC5--HUB--PC3
          |
         PC4
    """
    print("\n" + "=" * 60)
    print("TEST 2: STAR TOPOLOGY WITH HUB")
    print("=" * 60)
    print("Method: send_broadcast() - hub broadcasts to all\n")
    
    # Create hub and 5 PCs
    hub = Hub("CentralHub", ports=5)
    pcs = [EndDevice(f"PC{i}") for i in range(1, 6)]
    
    # Connect all PCs to hub (each PC has ONLY hub in connections)
    for pc in pcs:
        hub.connect(pc)
    
    # Show connections
    print(f"\n--- Connection Status ---")
    for pc in pcs:
        print(f"{pc.name} connects to: {[d.name for d in pc.connections]}")
    
    # PC1 sends to PC3 using BROADCAST (goes to hub, hub repeats to all)
    print(f"\n--- PC1 sends to PC3 (via hub broadcast) ---")
    msg = Packet("PC1", "PC3", "Hello PC3!")
    pcs[0].send_broadcast(msg)  # PC1 sends to its only connection (hub)
    
    # Check who got it
    print(f"\n--- DELIVERY REPORT ---")
    for pc in pcs:
        print(f"{pc.name}: {len(pc.inbox)} message(s)")
    
    # Verification
    passed = (
        len(pcs[0].inbox) == 0 and  # PC1 got nothing back
        len(pcs[1].inbox) == 0 and  # PC2 ignored
        len(pcs[2].inbox) == 1 and  # PC3 accepted!
        len(pcs[3].inbox) == 0 and  # PC4 ignored
        len(pcs[4].inbox) == 0      # PC5 ignored
    )
    
    print(f"\n{'✓ TEST PASSED' if passed else '✗ TEST FAILED'}")
    print("(Hub broadcast to all, only PC3 kept it)")


In [17]:
# ============================================================
# TEST CASE 3: Try Direct Send in Hub Topology (Should Fail!)
# ============================================================

def test_direct_in_hub_topology():
    """
    Educational test: What happens if you try direct send in a hub topology?
    Should FAIL because PCs aren't directly connected to each other!
    """
    print("\n" + "=" * 60)
    print("TEST 3: DIRECT SEND IN HUB TOPOLOGY (SHOULD FAIL)")
    print("=" * 60)
    print("This shows why you need the right method for your topology!\n")
    
    hub = Hub("Hub", ports=3)
    pc1 = EndDevice("PC1")
    pc2 = EndDevice("PC2")
    
    # Connect to hub (PC1 and PC2 are NOT directly connected!)
    hub.connect(pc1)
    hub.connect(pc2)
    
    print(f"\n--- Trying direct send from PC1 to PC2 ---")
    print(f"PC1's connections: {[d.name for d in pc1.connections]}")
    print(f"(PC2 is NOT in this list! Only Hub is)")
    
    msg = Packet("PC1", "PC2", "Direct message?")
    success = pc1.send_direct(msg, pc2)
    
    if not success:
        print(f"\n✓ CORRECT: Direct send failed!")
        print(f"  Reason: PC1 and PC2 aren't directly connected.")
        print(f"  They only share a connection through the Hub.")
        print(f"  Solution: Use send_broadcast() instead!")
